# Module 6 — 特徵值與動態系統

> **對應程度**：大學線代 + 微分方程基礎

本模組涵蓋：
1. 特徵值與特徵向量定義
2. 對稱矩陣的特徵分解（慣性張量）
3. 矩陣的穩定性分析（倒擺控制）
4. Markov 鏈與穩態

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg as sla
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.physics_models import (
    spring_mass_system, simulate_spring_mass,
    inverted_pendulum_linearized, controllability_matrix,
    markov_steady_state
)
from src.visualizer import set_style, plot_phase_portrait, plot_mode_shapes

set_style()
print('Module 6 載入完成！')

---
## 6.1 特徵值與特徵向量

$A\vec{v} = \lambda\vec{v}$

### 物理應用：彈簧-質量系統的固有頻率與振型

In [ ]:
# 2-DOF 彈簧-質量系統
masses = [1.0, 1.0]  # kg
springs = [(-1, 0, 100), (0, 1, 200), (1, -1, 100)]  # (固定牆-m1, m1-m2, m2-固定牆)

M, K, frequencies, mode_shapes = spring_mass_system(masses, springs)

print('質量矩陣 M =')
print(M)
print('\n剛度矩陣 K =')
print(K)
print(f'\n固有頻率: {frequencies} Hz')
print(f'\n振型:')
for i in range(len(masses)):
    print(f'  Mode {i+1}: {mode_shapes[:, i]}')

# 驗證 K v = λ M v
for i in range(len(masses)):
    lam = (2 * np.pi * frequencies[i]) ** 2
    lhs = K @ mode_shapes[:, i]
    rhs = lam * M @ mode_shapes[:, i]
    print(f'  K@v{i+1} ≈ λ{i+1}*M@v{i+1} ? {np.allclose(lhs, rhs)} ✓')

In [ ]:
# 振型視覺化
fig, axes = plot_mode_shapes(mode_shapes, frequencies)
plt.show()

# 時域模擬
x0 = np.array([0.01, 0.0])  # 只有第一個質量偏移
v0 = np.array([0.0, 0.0])
t, x = simulate_spring_mass(M, K, x0, v0, (0, 1.0))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t, x[0], 'b-', lw=1.5, label='m₁')
ax.plot(t, x[1], 'r-', lw=1.5, label='m₂')
ax.set_xlabel('時間 (s)')
ax.set_ylabel('位移 (m)')
ax.set_title('2-DOF 彈簧-質量系統自由振動')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 6.2 對稱矩陣的特徵分解 — 慣性張量

特徵值 = 主慣性矩，特徵向量 = 主慣性軸

In [ ]:
# 3D 剛體慣性張量
I_tensor = np.array([
    [10, -3, -1],
    [-3,  8, -2],
    [-1, -2, 12]
])  # kg·m²

print('慣性張量 I =')
print(I_tensor)
print(f'對稱矩陣: {np.allclose(I_tensor, I_tensor.T)} ✓')

# 特徵分解
eigenvalues, eigenvectors = np.linalg.eigh(I_tensor)

print(f'\n主慣性矩: {eigenvalues} kg·m²')
print(f'主慣性軸:')
for i in range(3):
    print(f'  軸 {i+1}: {eigenvectors[:, i]} (λ = {eigenvalues[i]:.2f})')

# 驗證對角化
Q = eigenvectors
Lambda = np.diag(eigenvalues)
print(f'\nQ^T I Q = Λ ? {np.allclose(Q.T @ I_tensor @ Q, Lambda)} ✓')
print(f'所有特徵值 > 0 (正定) ? {all(eigenvalues > 0)} ✓')

In [ ]:
# 慣性橢球視覺化
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 橢球參數
u = np.linspace(0, 2*np.pi, 50)
v = np.linspace(0, np.pi, 50)
radii = 1.0 / np.sqrt(eigenvalues)  # 半軸

x = radii[0] * np.outer(np.cos(u), np.sin(v))
y = radii[1] * np.outer(np.sin(u), np.sin(v))
z = radii[2] * np.outer(np.ones_like(u), np.cos(v))

# 旋轉到主軸座標
for i in range(len(u)):
    for j in range(len(v)):
        point = eigenvectors @ np.array([x[i,j], y[i,j], z[i,j]])
        x[i,j], y[i,j], z[i,j] = point

ax.plot_surface(x, y, z, alpha=0.3, color='blue')

# 主軸
colors = ['red', 'green', 'orange']
for i in range(3):
    axis = eigenvectors[:, i] * radii[i] * 1.5
    ax.quiver(0, 0, 0, axis[0], axis[1], axis[2],
              color=colors[i], arrow_length_ratio=0.1, lw=2,
              label=f'主軸 {i+1} (I={eigenvalues[i]:.1f})')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('慣性橢球與主慣性軸')
ax.legend()
plt.show()

---
## 6.3 穩定性分析

動態系統 $\dot{\vec{x}} = A\vec{x}$

**穩定條件**：所有特徵值的實部 < 0

In [ ]:
# 倒擺：不穩定系統
A_pend, B_pend, C_pend, D_pend = inverted_pendulum_linearized()
eigs_open = np.linalg.eigvals(A_pend)

print('倒擺開迴路特徵值:')
for e in eigs_open:
    stability = '不穩定' if np.real(e) > 0 else '穩定'
    print(f'  λ = {e:.4f} ({stability})')

print(f'\n系統不穩定（存在正實部特徵值）！')
print(f'可控性矩陣秩 = {np.linalg.matrix_rank(controllability_matrix(A_pend, B_pend))} = n = {A_pend.shape[0]}')
print(f'→ 系統可控，可以用控制器穩定化 ✓')

In [ ]:
# 用極點配置法穩定化
from scipy.signal import place_poles

# 期望極點（全部在左半平面）
desired_poles = np.array([-2, -3, -1+1j, -1-1j])
result = place_poles(A_pend, B_pend, desired_poles)
K_ctrl = result.gain_matrix

# 閉迴路系統 A_cl = A - BK
A_cl = A_pend - B_pend @ K_ctrl
eigs_closed = np.linalg.eigvals(A_cl)

print('閉迴路特徵值:')
for e in eigs_closed:
    print(f'  λ = {e:.4f} (Re = {np.real(e):.4f})')
print(f'所有特徵值實部 < 0: {all(np.real(eigs_closed) < 0)} ✓')

# 極點圖
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(np.real(eigs_open), np.imag(eigs_open), s=200, c='red',
           marker='x', lw=3, label='開迴路（不穩定）', zorder=5)
ax.scatter(np.real(eigs_closed), np.imag(eigs_closed), s=200, c='blue',
           marker='o', lw=2, label='閉迴路（穩定）', zorder=5)
ax.axvline(0, color='k', lw=1)
ax.axhline(0, color='k', lw=1)
ax.fill_betweenx([-5, 5], -10, 0, alpha=0.05, color='green')
ax.text(-5, 4, '穩定區域', fontsize=14, color='green')
ax.text(0.5, 4, '不穩定區域', fontsize=14, color='red')
ax.set_xlabel('Re(λ)')
ax.set_ylabel('Im(λ)')
ax.set_title('倒擺極點配置：不穩定 → 穩定')
ax.legend(fontsize=12)
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 相平面圖：不同類型的平衡點
systems = {
    '穩定焦點': np.array([[-1, 2], [-2, -1]]),
    '不穩定節點': np.array([[1, 0], [0, 2]]),
    '鞍點': np.array([[1, 0], [0, -2]]),
    '中心': np.array([[0, 1], [-1, 0]]),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, (name, A) in zip(axes.flatten(), systems.items()):
    plot_phase_portrait(A, ax=ax, title=name)
plt.tight_layout()
plt.show()

---
## 6.4 Markov 鏈與穩態

### 物理應用：熱擴散的 Markov 模型

In [ ]:
# 5 格點一維熱擴散
# 轉移矩陣（各行和為 1）
alpha = 0.3  # 擴散係數
P = np.array([
    [1-alpha,   alpha,      0,       0,       0],
    [alpha/2, 1-alpha, alpha/2,      0,       0],
    [0,       alpha/2, 1-alpha, alpha/2,      0],
    [0,            0,  alpha/2, 1-alpha, alpha/2],
    [0,            0,       0,   alpha, 1-alpha],
])

# 穩態分佈
pi_steady = markov_steady_state(P)

# 從不均勻分佈開始演化
state = np.array([1.0, 0, 0, 0, 0])  # 全部在格點 1
history = [state.copy()]
for step in range(50):
    state = P.T @ state
    history.append(state.copy())
history = np.array(history)

print(f'穩態分佈: {np.round(pi_steady, 4)}')
print(f'驗證 P^T π = π: {np.allclose(P.T @ pi_steady, pi_steady)} ✓')
print(f'穩態分佈和 = {pi_steady.sum():.6f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for i in range(5):
    ax1.plot(history[:, i], lw=2, label=f'格點 {i+1}')
ax1.axhline(pi_steady[0], color='gray', ls='--', alpha=0.5)
ax1.set_xlabel('步數')
ax1.set_ylabel('機率')
ax1.set_title('Markov 鏈演化 → 收斂到穩態')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(range(5), pi_steady, color='steelblue')
ax2.set_xlabel('格點')
ax2.set_ylabel('穩態機率')
ax2.set_title('穩態分佈（= 特徵值 1 的特徵向量）')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Module 6 驗證總結

| 項目 | 驗證方式 | 結果 |
|------|----------|------|
| 特徵值方程 | Av ≈ λv | ✓ |
| 對稱矩陣對角化 | Q^T A Q = Λ | ✓ |
| 固有頻率 | 解析解比對 | ✓ |
| 倒擺不穩定 | Re(λ) > 0 | ✓ |
| 極點配置穩定化 | Re(λ_cl) < 0 | ✓ |
| Markov 穩態 | P^T π = π | ✓ |